# ROM-FlameBench: a simple walkthrough

This notebook runs three complete experiments and one hyperparameter search:

1. Prepare and load the data.
2. Build the datasets for the compressor and the forecaster.
3. **POD-ARX**: linear compression and a linear forecaster.
4. **POD-LSTM**: the same compression and a recurrent forecaster trained on multi-step rollouts.
5. **CAE-ARX**: a convolutional autoencoder and a linear forecaster.
6. **HPO of POD-ARX**: tune the forecaster and the history lengths on validation data.
7. Compare the results.

Run the cells in order, from the repository root, using the project's Python environment.
The fits use the real dataset and can take time; the CAE needs a GPU to be quick.

**Confirmed:** snapshot zero is the initial steady state.
**TODO:** add physical cell volumes. Until then, we evaluate fields only; integrated
heat release and its gain/phase metrics are disabled.

## 1. Data preparation

Raw archives contain `data` with shape **(cell, field, time)**. Preparation maps the mesh cells
to pixels and writes float32 `.npy` images with shape **(time, field, 206, 104)**. Pixels
without a cell are invalid: a mask marks the valid ones, and every error below uses valid
pixels only. The images are read from disk in small batches. Existing prepared files are reused.

The supplied `phi` files contain the dimensionless forcing: $U(t)=U_{base}\phi(t)$.
The metadata defines filenames, field order, units and the sampling interval.

In [1]:
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from DataProcessing.prepare import prepare
from DataProcessing.metadata import load_metadata
from DataProcessing.Dataset import CompressorDataset, ForecasterDataset
from utils import seed_everything

device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 42
seed_everything(seed)
torch.set_num_threads(4)

metadata_path = Path("Data/metadata.json")
raw_directory = Path("Data/Raw")  # Server: Path("/srv/mlg/shared/FlameBench/Raw")
output = Path("Experiments/Results") / time.strftime("walkthrough_%Y%m%dT%H%M%S")
print("Output directory:", output)

Nx = 9               # Past states besides the current one: 10 in total.
Ni = 4               # Past forcing values besides the next one: 5 in total.
horizon = 10         # K: recursive steps of the LSTM training loss.
K_eval = 50          # Recursive steps of the validation forecasts.
rank = 16            # Latent size, fixed for every compressor.
batch_size = 64
validation_fraction = 0.2
blocks = 20          # Each training sweep is cut into this many blocks in time.
lstm_epochs = 100
cae_epochs = 20
cae_channels = [16, 32, 64]
trials = 2           # HPO trials per stage.
hpo_fixed_dataset = {}  # Dataset values kept fixed during HPO; the others are tuned.

# .env can enable online W&B logging for the FireMark team.
logging_config = {"logging": {"wandb": {"mode": "offline", "entity": "FireMark"}}}
output.mkdir(parents=True, exist_ok=True)

Output directory: Experiments/Results/walkthrough_20260922T170654


In [2]:
prepare(metadata_path, raw_directory)
metadata = load_metadata(metadata_path)

print("Fields:", metadata["fields"])
print("Sampling interval:", metadata["dt"], "seconds")
for case in metadata["cases"]:
    print(case["split"], case["name"])

Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Images/Training/sineSweep_f1_f80_A02.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Images/Training/phi_sineSweep_f1_f80_A02.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Images/Training/sineSweep_f1_f80_A04.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Images/Training/phi_sineSweep_f1_f80_A04.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Images/Test/sine_f10_A03.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Images/Test/phi_sine_f10_A03.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Images/Test/sine_f10_A05.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Images/Test/phi_sine_f10_A05.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Images/Test/sine_f40_A03.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Images/Test/phi_sine_f40_A03.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBenc

## 2. Datasets

Each training sweep is cut in time into **`blocks` equal blocks**. A fraction
`validation_fraction` of the blocks, evenly spaced, is validation; the others are training.
Both partitions therefore cover the whole sweep, low and high frequencies. Consecutive blocks
form one segment, and samples never cross a segment edge, so no frame is used by both
partitions. Test simulations are kept separate.

There are two kinds of dataset, and both use exactly the same split:

- A `CompressorDataset` sample is **one frame**.
- A `ForecasterDataset` sample is a **window** of `max(Nx, Ni) + 1` rows ending at time $t$.
  Row $s$ holds the state $x(s)$ and the forcing deviation $\phi(s+1)-1$, so the last row pairs
  $x(t)$ with the known next forcing. **Nx and Ni are independent**: states older than the last
  `Nx + 1` rows and forcing older than the last `Ni + 1` rows are set to zero. The targets are
  the next `horizon` states $x(t+1),\ldots,x(t+K)$.

The `Dataset` defines samples; a `DataLoader` groups them into batches.

In [3]:
split = dict(validation_fraction=validation_fraction, blocks=blocks)
train_compressor_ds = CompressorDataset(metadata, "train", **split)
train_forecaster_ds = ForecasterDataset(metadata, "train", Nx=Nx, Ni=Ni, **split)
val_compressor_ds = CompressorDataset(metadata, "validation", **split)
val_forecaster_ds = ForecasterDataset(metadata, "validation", Nx=Nx, Ni=Ni, **split)

test_ds = ForecasterDataset(metadata, "test", Nx=Nx, Ni=Ni)

sample = train_forecaster_ds[0]
print("Training segments:", [(case["name"], start, stop) for case, start, stop in train_compressor_ds.segments])
print("Training frames:", len(train_compressor_ds))
print("Training windows:", len(train_forecaster_ds))
print("States:", tuple(sample["states"].shape))  # (max(Nx, Ni) + 1, fields, height, width)
print("Forcing:", sample["forcing"].tolist())   # phi(s+1) - 1 for each row
print("Target:", tuple(sample["target"].shape))  # (horizon, fields, height, width)

Training segments: [('sineSweep_f1_f80_A02', 0, 400), ('sineSweep_f1_f80_A02', 600, 1400), ('sineSweep_f1_f80_A02', 1600, 2400), ('sineSweep_f1_f80_A02', 2600, 3400), ('sineSweep_f1_f80_A02', 3600, 4001), ('sineSweep_f1_f80_A04', 0, 400), ('sineSweep_f1_f80_A04', 600, 1400), ('sineSweep_f1_f80_A04', 1600, 2400), ('sineSweep_f1_f80_A04', 2600, 3400), ('sineSweep_f1_f80_A04', 3600, 4001)]
Training frames: 6402
Training windows: 6302
States: (10, 11, 206, 104)
Forcing: [0.0, 0.0, 0.0, 0.0, 0.0, 0.003794550895690918, 0.004431724548339844, 0.005070328712463379, 0.005710244178771973, 0.006351470947265625]
Target: (1, 11, 206, 104)


## 3. POD-ARX

The fields have different units and scales. First, we estimate a mean and standard deviation
for each field **using training frames only and excluding empty pixels**. The scaler is fitted
once and shared by every experiment; the datasets then return scaled frames.

**POD** flattens each scaled image into the vector of its valid pixels and keeps the first
`rank` modes of the training frames (legacy randomized SVD: oversampling 20, 7 iterations,
seed 42). `encode` maps a frame to `rank` coefficients and `decode` maps them back. The
reconstruction error measures compression alone.

In [4]:
from DataProcessing.scaling import FeatureScaler
from Baselines.OrderReduction.Linear.POD import POD
from Experiments.evaluation import reconstruction_error

scaler = FeatureScaler(train_compressor_ds.mask).fit(DataLoader(train_compressor_ds, batch_size=batch_size))
train_compressor_ds = CompressorDataset(metadata, "train", scaler=scaler, **split)
val_compressor_ds = CompressorDataset(metadata, "validation", scaler=scaler, **split)

pod = POD(rank=rank, batch_size=batch_size).fit(train_compressor_ds)
print("POD validation reconstruction MSE:", reconstruction_error(pod, val_compressor_ds))

POD validation reconstruction MSE: 0.017777498940197825


Given a compressor, a `ForecasterDataset` encodes its frames once and keeps the small latent
trajectories in memory. The helpers below are shared by all experiments:

- `latent_windows` builds the training windows and the validation windows (`K_eval` targets).
- `validation_mse` is the **selection objective**: from each validation window, forecast
  `K_eval` steps recursively (each prediction is fed back), decode, and compare with the scaled
  frames on valid pixels. Windows are `K_eval` apart, so each frame is compared once.
- `test_rollouts` forecasts each test simulation from its steady first snapshot to the end.

**ARX** is a ridge regression of the increment $z_{t+1}-z_t$ on the latent history, the forcing
history and a constant:

$$\widehat z_{t+1} = z_t + W\,[z_{t-N_x},\ldots,z_t,\ \phi_{t+1-N_i}-1,\ldots,\phi_{t+1}-1,\ 1].$$

`alpha` controls the ridge penalty. ARX has a closed-form fit on one-step targets.

In [5]:
from Baselines.Forecast.Classical.ARX import ARX
from Experiments.evaluation import validation_error, evaluate


scores, tests = {}, {}
train_forecaster_dataset = ForecasterDataset(metadata, "train", horizon=1, Nx=Nx, Ni=Ni, scaler=scaler,compressor=pod, **split)
val_forecaster_dataset = ForecasterDataset(metadata, "validation",horizon=20,stride=20, Nx=Nx, Ni=Ni, scaler=scaler, **split)

arx = ARX(Nx=Nx, Ni=Ni, alpha=1e-4).fit(DataLoader(train_forecaster_dataset, batch_size=batch_size))
scores["pod_arx"] = validation_error(arx, pod, val_forecaster_dataset)
tests["pod_arx"] = evaluate(arx, test_ds, pod, scaler, output / "pod_arx" )
print("POD-ARX validation field MSE:", scores["pod_arx"])

POD-ARX validation field MSE: 0.021244048070313132


Each test rollout starts from **snapshot zero**, repeated to fill the history. No later
ground-truth field is supplied to the model. Forcing before time zero is the unforced value
($\phi=1$). At each step the forecaster predicts the next latent state, the compressor decodes
it, and the prediction is fed back as the newest state.

For each field, NRMSE is the RMSE divided by the ground-truth standard deviation, pooling all
evaluated cells and times in that case. The mean NRMSE averages the 11 field scores. Snapshot
zero is excluded because it was given to the model.

## 4. POD-LSTM

The **LSTM** reads the same rows $[z_s,\ \phi_{s+1}-1]$ and predicts the increment of the last
state. It is trained on **multi-step rollouts**: each training window has `horizon` targets,
the prediction of each step is fed back, and the loss is

$$(1-w)\,L_1 + w\,\tfrac{1}{K}\textstyle\sum_{k=1}^{K} L_k .$$

Early stopping uses the same loss on the validation windows, which have `K_eval` targets, so
the model is trained on $K$ steps and checked on longer forecasts.

In [6]:
from Baselines.Forecast.DL.networks import LSTM

train_forecaster_dataset = ForecasterDataset(metadata, "train", horizon=10, Nx=Nx, Ni=Ni, scaler=scaler,compressor=pod, **split)
val_forecaster_dataset = ForecasterDataset(metadata, "validation",horizon=20,stride=20, Nx=Nx, Ni=Ni, scaler=scaler, **split)
lstm = LSTM(input_size=pod.rank + 1, output_size=pod.rank, Nx=Nx, Ni=Ni, hiddens=[64], epochs=lstm_epochs, patience=20, device=device)
lstm.fit(DataLoader(train_forecaster_dataset, batch_size=batch_size, shuffle=True), DataLoader(ForecasterDataset(metadata, "validation",horizon=20,stride=20, Nx=Nx, Ni=Ni, scaler=scaler,compressor=pod, **split), batch_size=batch_size))
scores["pod_lstm"] = validation_error(lstm, pod, val_forecaster_dataset)
tests["pod_lstm"] = evaluate(arx, test_ds, pod, scaler, output / "pod_lstm" )
print("POD-LSTM validation field MSE:", scores["pod_lstm"])

POD-LSTM validation field MSE: 0.029816403001105326


## 5. CAE-ARX

The **CAE** compresses each image with convolutions (`channels` gives the width of each
level), then a linear layer to `rank` values. The decoder mirrors the encoder, so the output
has exactly the input shape. Training shuffles the training frames every epoch and keeps the
epoch with the lowest validation reconstruction error. The ARX forecaster is the same as in
section 3, now on the CAE latents.

In [7]:
from Baselines.OrderReduction.DL.CAE import CAE

cae = CAE(rank=rank, channels=cae_channels, epochs=cae_epochs, batch_size=16, device=device)
cae.fit(train_compressor_ds, validation=val_compressor_ds)
print("CAE validation reconstruction MSE:", reconstruction_error(cae, val_compressor_ds))

train_forecaster_dataset = ForecasterDataset(metadata, "train", horizon=10, Nx=Nx, Ni=Ni, scaler=scaler,compressor=cae, **split)
val_forecaster_dataset = ForecasterDataset(metadata, "validation",horizon=20,stride=20, Nx=Nx, Ni=Ni, scaler=scaler, **split)
cae_arx = ARX(Nx=Nx, Ni=Ni, alpha=1e-4).fit(DataLoader(train_forecaster_dataset, batch_size=batch_size))
scores["cae_arx"] =validation_error(cae_arx, cae, val_forecaster_dataset)
tests["cae_arx"] = evaluate(cae_arx, test_ds, cae, scaler, output / "cae_arx" )
print("CAE-ARX validation field MSE:", scores["cae_arx"])

KeyboardInterrupt: 

## 6. HPO of POD-ARX

`Experiments.HPO.optimize` receives only the classes and a config. Every class lists its search
space in `hyperparameters_ranges`, and `build` creates an object from chosen values. Values
written in the config are **fixed**; the other entries of the ranges are **tuned** with Optuna.
Here the POD rank is fixed, and the search tunes ARX's `alpha` together with the dataset's `Nx`
and `Ni`. ARX fixes the dataset `horizon` to 1, because its closed-form fit is one-step.

With a compressor and a forecaster the search runs in stages, each keeping the best result of
the stage before:

1. the compressor, on the validation reconstruction error (POD has nothing to tune: one fit);
2. the forecaster and the dataset, with the compressor frozen, on the validation field MSE;
3. joint training of a neural forecaster and an autoencoder (not used by POD-ARX).

`optimize` returns the best config. `fit` then trains it from scratch, and `test` evaluates it.

In [ ]:
from Experiments.HPO import optimize
from Experiments.run import fit, test

hpo_config = {
    "metadata": str(metadata_path), "output": str(output), "run_name": "hpo_pod_arx", "seed": seed,
    "device": device, "validation_fraction": validation_fraction, "blocks": blocks, "K_eval": K_eval,
    "trials": trials, "batch_size": batch_size,
    "compressor": {"name": "pod", "rank": rank},
    "dataset": dict(hpo_fixed_dataset),
    "forecaster": {"name": "arx"},
    "evaluation": {"heat_release": False},
    **logging_config,
}
best = optimize(hpo_config, POD, ARX)
print("Best config:", best)

best_config = {**hpo_config, **best}
scores["hpo_pod_arx"] = fit(best_config)
tests["hpo_pod_arx"] = test(best_config)

## 7. Results

The validation field MSE is the selection objective (scaled fields, valid pixels, `K_eval`-step
forecasts). The test columns give the mean NRMSE of each test simulation, forecast from its
first snapshot to the end.

In [ ]:
cases = list(tests["pod_arx"])
print(f"{'experiment':<14}{'validation MSE':>16}" + "".join(f"{case:>16}" for case in cases))
for name, results in tests.items():
    print(f"{name:<14}{scores[name]:>16.4g}" + "".join(f"{results[case]['mean_nrmse']:>16.4g}" for case in cases))

Finally, send the scalar results to **TensorBoard and W&B**. W&B defaults to offline here; the
project's `.env` can set `WANDB_MODE=online` and provide the API key for `FireMark`.
Credentials are not saved with these settings. TensorBoard can read the results with
`tensorboard --logdir Experiments/Results`.

In [ ]:
from Experiments.logging import ExperimentLogger

logger = ExperimentLogger(output / "summary", {"run_name": output.name, "seed": seed, **logging_config})
try:
    for step, (name, results) in enumerate(tests.items()):
        values = {f"{name}/validation_field_mse": scores[name]}
        for case, metrics in results.items():
            values[f"{name}/test/{case}/mean_nrmse"] = metrics["mean_nrmse"]
        logger.log(values, step=step)
finally:
    logger.close()